# A3. AI 辅助广告优化：搜索词报告分析实战

> 配套模块：[A3 广告优化](../paths/a-operators/a3-advertising.md)
>
> 本 Notebook 演示如何用 Python + AI 分析 Amazon 搜索词报告，找出高 ROAS 词、浪费词和隐藏机会。
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a3-advertising.ipynb)

## 1. 准备工作

本 Notebook 不需要 API Key，使用纯 Python 进行数据处理。
AI 分析部分提供 Prompt 模板，你可以复制到 ChatGPT/Claude 中使用。

In [ ]:
import pandas as pd
import numpy as np

print('环境准备完成 ✅')

## 2. 模拟搜索词报告数据

实际使用时，从 Amazon Seller Central → 广告 → 搜索词报告 下载 CSV。
这里用模拟数据演示分析流程。

In [ ]:
# 模拟 Amazon 搜索词报告
np.random.seed(42)

search_terms = [
    'portable neck fan', 'neck fan usb', 'hands free fan', 'personal fan',
    'neck fan rechargeable', 'bladeless neck fan', 'cooling fan portable',
    'fan for running', 'outdoor neck fan', 'best neck fan 2026',
    'mini fan', 'desk fan usb', 'clip on fan', 'stroller fan',
    'neck fan for kids', 'neck fan quiet', 'neck fan long battery',
    'wearable fan', 'travel fan portable', 'neck air conditioner',
    'cheap fan', 'fan amazon', 'cooling device', 'summer gadgets',
    'neck fan pink', 'neck fan black', 'fan with led', 'sports fan neck',
    'construction fan', 'warehouse fan'
]

data = {
    'Search Term': search_terms,
    'Impressions': np.random.randint(500, 50000, len(search_terms)),
    'Clicks': np.random.randint(5, 500, len(search_terms)),
    'Spend': np.round(np.random.uniform(1, 200, len(search_terms)), 2),
    'Orders': np.random.randint(0, 30, len(search_terms)),
    'Sales': np.round(np.random.uniform(0, 600, len(search_terms)), 2)
}

df = pd.DataFrame(data)
# 确保逻辑一致性
df.loc[df['Orders'] == 0, 'Sales'] = 0
df['CTR'] = np.round(df['Clicks'] / df['Impressions'] * 100, 2)
df['CPC'] = np.round(df['Spend'] / df['Clicks'], 2)
df['ACOS'] = np.where(df['Sales'] > 0, np.round(df['Spend'] / df['Sales'] * 100, 1), np.inf)
df['ROAS'] = np.where(df['Spend'] > 0, np.round(df['Sales'] / df['Spend'], 2), 0)
df['CVR'] = np.where(df['Clicks'] > 0, np.round(df['Orders'] / df['Clicks'] * 100, 2), 0)

print(f'搜索词数量: {len(df)}')
print(f'总花费: ${df["Spend"].sum():.2f}')
print(f'总销售: ${df["Sales"].sum():.2f}')
print(f'整体 ROAS: {df["Sales"].sum() / df["Spend"].sum():.2f}x')
df.head(10)

## 3. 搜索词分类：四象限分析

把搜索词按 ROAS 和花费分成四类：
- 🟢 **明星词**：高 ROAS + 高花费 → 加大投入
- 🔵 **潜力词**：高 ROAS + 低花费 → 提高出价获取更多流量
- 🟡 **观察词**：低 ROAS + 低花费 → 继续观察或优化
- 🔴 **浪费词**：低 ROAS + 高花费 → 降低出价或否定

In [ ]:
# 设置阈值
ROAS_THRESHOLD = 3.0  # ROAS >= 3 为高
SPEND_THRESHOLD = df['Spend'].median()  # 花费中位数为分界

def classify_term(row):
    if row['Orders'] == 0 and row['Spend'] > 10:
        return '🔴 浪费词（零转化）'
    elif row['ROAS'] >= ROAS_THRESHOLD and row['Spend'] >= SPEND_THRESHOLD:
        return '🟢 明星词'
    elif row['ROAS'] >= ROAS_THRESHOLD and row['Spend'] < SPEND_THRESHOLD:
        return '🔵 潜力词'
    elif row['ROAS'] < ROAS_THRESHOLD and row['Spend'] >= SPEND_THRESHOLD:
        return '🔴 浪费词'
    else:
        return '🟡 观察词'

df['分类'] = df.apply(classify_term, axis=1)

print('=== 搜索词分类统计 ===')
print(df['分类'].value_counts())
print()

for cat in ['🟢 明星词', '🔵 潜力词', '🔴 浪费词', '🔴 浪费词（零转化）', '🟡 观察词']:
    subset = df[df['分类'] == cat]
    if len(subset) > 0:
        print(f'\n--- {cat} ({len(subset)} 个) ---')
        print(subset[['Search Term', 'Spend', 'Sales', 'ROAS', 'Orders']].to_string(index=False))

## 4. 否定词候选列表

自动找出应该否定的搜索词：花费 > $10 但零转化，或 ROAS < 1。

In [ ]:
# 否定词候选
negative_candidates = df[
    ((df['Orders'] == 0) & (df['Spend'] > 10)) |
    ((df['ROAS'] < 1) & (df['ROAS'] > 0) & (df['Spend'] > 20))
].sort_values('Spend', ascending=False)

print(f'=== 建议否定的搜索词（{len(negative_candidates)} 个）===')
print(f'否定这些词可节省: ${negative_candidates["Spend"].sum():.2f}')
print()
print(negative_candidates[['Search Term', 'Spend', 'Orders', 'ROAS']].to_string(index=False))

## 5. 出价优化建议

基于 ROAS 给出每个搜索词的出价调整建议。

In [ ]:
def bid_suggestion(row):
    if row['Orders'] == 0:
        return '否定或暂停'
    elif row['ROAS'] >= 5:
        return f'提高出价 +20%（当前 CPC ${row["CPC"]:.2f}）'
    elif row['ROAS'] >= 3:
        return f'维持当前出价（CPC ${row["CPC"]:.2f}）'
    elif row['ROAS'] >= 1.5:
        return f'降低出价 -15%（当前 CPC ${row["CPC"]:.2f}）'
    else:
        return f'降低出价 -30% 或否定（当前 CPC ${row["CPC"]:.2f}）'

df['出价建议'] = df.apply(bid_suggestion, axis=1)

print('=== 出价优化建议 ===')
for _, row in df.sort_values('Spend', ascending=False).head(15).iterrows():
    print(f'{row["Search Term"]:30s} | ROAS {row["ROAS"]:5.2f}x | {row["出价建议"]}')

## 6. 生成 AI 分析 Prompt

把数据整理成 Prompt，复制到 ChatGPT/Claude 获取更深入的分析。

In [ ]:
# 生成 AI Prompt
top_terms = df.nlargest(20, 'Spend')[['Search Term', 'Impressions', 'Clicks', 'Spend', 'Orders', 'Sales', 'ROAS', 'ACOS']]

prompt = f"""你是一个 Amazon PPC 广告优化专家。

以下是我的搜索词报告数据（过去 14 天，按花费排序 Top 20）：

总花费: ${df['Spend'].sum():.2f}
总销售: ${df['Sales'].sum():.2f}
整体 ROAS: {df['Sales'].sum() / df['Spend'].sum():.2f}x

{top_terms.to_markdown(index=False)}

请分析：
1. 哪些是高价值词？应该提高出价多少？
2. 哪些词在浪费预算？应该否定还是降低出价？
3. 有没有隐藏的长尾机会？
4. 整体广告策略建议（预算分配、新关键词方向）
"""

print('=== 复制以下 Prompt 到 ChatGPT/Claude ===')
print(prompt)

## 7. 总结

本 Notebook 演示了：
1. 搜索词四象限分类（明星词/潜力词/浪费词/观察词）
2. 自动生成否定词候选列表
3. 基于 ROAS 的出价优化建议
4. 自动生成 AI 分析 Prompt

**下一步**：
- 用你自己的搜索词报告 CSV 替换模拟数据
- 把生成的 Prompt 复制到 ChatGPT/Claude 获取更深入的分析
- 参考 [A3 广告优化](../paths/a-operators/a3-advertising.md) 了解完整的广告优化方法论

---
📎 **来源**: [ecommerce-ai-roadmap](https://github.com/kangise/ecommerce-ai-roadmap)